[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/projects/case1_missing_revenue/day_01.ipynb)

# Case 1, day 1: The missing revenue

*About 60 minutes.*

## Setup

Run this cell once. It loads the data and gives you `run(...)`.

In [ ]:
# Setup: loads the real sales data of a UK online gift shop (Dec 2010 to Dec 2011, 541,909 rows) into a
# small SQL database, and gives you run(query) to ask it questions. Run this cell once.
import os, sqlite3, urllib.request
import pandas as pd

URL = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/online_retail.csv.gz"
path = next((p for p in ["../../exam_prep/data/online_retail.csv.gz", "../data/online_retail.csv.gz",
                         "online_retail.csv.gz"] if os.path.exists(p)), None)
if path is None:
    print("downloading the data (about 7 MB)...")
    path = "online_retail.csv.gz"
    urllib.request.urlretrieve(URL, path)

df = pd.read_csv(path, dtype={"InvoiceNo": str, "StockCode": str})
df.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price",
              "customer_id", "country"]
df["customer_id"] = df["customer_id"].astype("Int64")
db = sqlite3.connect(":memory:")
df.to_sql("sales", db, index=False)

def run(query):
    """Runs a SQL query on the database and shows the result as a table."""
    if not query.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    return pd.read_sql_query(query, db)

print("Ready: table sales with", len(df), "rows")

## The briefing

You just started as the data analyst at a small UK online shop that sells gifts and home decoration.
On Monday morning the CEO sends you this message:

> "I just saw the dashboard. **December revenue is 433,668 pounds. November was 1,461,756.** That is a
> **70% drop**. Is the business collapsing? I need an answer by Friday."

Your job this week: find out what really happened, using SQL on the shop's sales data. Every day you learn a few
new SQL tools and use them to follow the next clue. Today you only **look at the evidence**: what is in the data,
which period it covers, and what looks strange.

**How each step works:** read the short explanation, write your query in the empty cell, then open the Hint if you
are stuck and the Answer to check. Try first, look second.

---
## Step 1: What does one row mean?

`SELECT` chooses columns, `FROM` chooses the table, and `LIMIT` shows only the first few rows. `*` means
"all columns". This is always the first query on new data: you cannot analyse what you have not looked at.

```sql
SELECT * FROM sales LIMIT 3;
```

### Your turn

Show the first **10 rows** of the `sales` table. Then answer for yourself: what does **one row** describe? An order, a customer, or something else?

In [ ]:
run("""

""")

<details><summary><b>Hint</b></summary>

`SELECT * FROM sales LIMIT 10`

</details>

<details><summary><b>Answer</b></summary>

```sql
SELECT * FROM sales LIMIT 10
```

Result:

| invoice_no | stock_code | description | quantity | invoice_date | unit_price | customer_id | country |
|---|---|---|---|---|---|---|---|
| 536365 | 85123A | WHITE HANGING HEART T-LIGHT HOLDER | 6 | 2010-12-01 08:26:00 | 2.55 | 17850 | United Kingdom |
| 536365 | 71053 | WHITE METAL LANTERN | 6 | 2010-12-01 08:26:00 | 3.39 | 17850 | United Kingdom |
| 536365 | 84406B | CREAM CUPID HEARTS COAT HANGER | 8 | 2010-12-01 08:26:00 | 2.75 | 17850 | United Kingdom |
| 536365 | 84029G | KNITTED UNION FLAG HOT WATER BOTTLE | 6 | 2010-12-01 08:26:00 | 3.39 | 17850 | United Kingdom |
| 536365 | 84029E | RED WOOLLY HOTTIE WHITE HEART. | 6 | 2010-12-01 08:26:00 | 3.39 | 17850 | United Kingdom |
| 536365 | 22752 | SET 7 BABUSHKA NESTING BOXES | 2 | 2010-12-01 08:26:00 | 7.65 | 17850 | United Kingdom |
| 536365 | 21730 | GLASS STAR FROSTED T-LIGHT HOLDER | 6 | 2010-12-01 08:26:00 | 4.25 | 17850 | United Kingdom |
| 536366 | 22633 | HAND WARMER UNION JACK | 6 | 2010-12-01 08:28:00 | 1.85 | 17850 | United Kingdom |
| 536366 | 22632 | HAND WARMER RED POLKA DOT | 6 | 2010-12-01 08:28:00 | 1.85 | 17850 | United Kingdom |
| 536367 | 84879 | ASSORTED COLOUR BIRD ORNAMENT | 32 | 2010-12-01 08:34:00 | 1.69 | 13047 | United Kingdom |

**Why:** Several rows share the same `invoice_no` (536365 appears many times). So one row is **one product line inside an invoice**, not one order. Counting rows would count product lines, not orders: an important detail for every number you report later.

> **Clue found:** One row = one product line of an invoice. An invoice (order) has many rows.

</details>

---
## Step 2: Which period does the data cover?

`ORDER BY column` sorts the result: `ASC` (smallest first, the default) or `DESC` (largest first).
Together with `LIMIT 1` it finds the first or the last value. Dates stored as text in the form
`YYYY-MM-DD HH:MM:SS` sort correctly, because the biggest unit comes first.

```sql
SELECT invoice_date FROM sales ORDER BY invoice_date DESC LIMIT 1;  -- the latest date
```

### Your turn

Find the **first** and the **last** `invoice_date` in the data. Look closely at the last one. What does it mean for the CEO's December number?

In [ ]:
run("""

""")

<details><summary><b>Hint</b></summary>

Two queries: `ORDER BY invoice_date ASC LIMIT 1` and `ORDER BY invoice_date DESC LIMIT 1`.

</details>

<details><summary><b>Answer</b></summary>

```sql
SELECT invoice_date FROM sales ORDER BY invoice_date DESC LIMIT 1
```

Result:

| invoice_date |
|---|
| 2011-12-09 12:50:00 |

**Why:** The first date is 2010-12-01 (run the ASC version too). The last is **2011-12-09**: the data stops on the 9th of December. So the dashboard compares a **full November** (30 days) with **only 9 days of December**. Part of the "drop" is simply missing days.

> **Clue found:** The data ends on 2011-12-09. December has only 9 days of data, November has 30.

</details>

---
## Step 3: Look only at the days that matter

`WHERE` keeps only the rows that match a condition. With text dates you can compare ranges:
`invoice_date >= '2011-12-01'` keeps December. `AND` combines conditions, and `COUNT(*)` counts the rows that are
left (we use counting properly tomorrow).

```sql
SELECT COUNT(*) FROM sales WHERE country = 'France';
```

### Your turn

How many rows (product lines) are there in **December 2011**? And in the **first 9 days of November 2011**?

In [ ]:
run("""

""")

<details><summary><b>Hint</b></summary>

`WHERE invoice_date >= '2011-12-01'` for December. For November use two conditions with `AND`: from `'2011-11-01'` and before `'2011-11-10'`.

</details>

<details><summary><b>Answer</b></summary>

```sql
SELECT
  (SELECT COUNT(*) FROM sales WHERE invoice_date >= '2011-12-01') AS dec_rows,
  (SELECT COUNT(*) FROM sales WHERE invoice_date >= '2011-11-01' AND invoice_date < '2011-11-10') AS nov_1_to_9_rows
```

Result:

| dec_rows | nov_1_to_9_rows |
|---|---|
| 25525 | 22170 |

**Why:** The first 9 days of December have even **more** product lines (25,525) than the first 9 days of November (22,170). That already hints that business did **not** collapse. (The answer combines two counts in one row with small subqueries; two separate queries are just as good.) Note `< '2011-11-10'`: "before the 10th" includes the whole 9th, even its afternoon.

> **Clue found:** Comparing equal periods (1 to 9 November vs 1 to 9 December), December is even busier.

</details>

---
## Step 4: Strange rows: negative quantities

Real data is messy. `WHERE quantity < 0` finds lines with a negative quantity. `LIKE` matches text
patterns: `%` means "any characters", so `invoice_no LIKE 'C%'` finds invoice numbers that start with C.

```sql
SELECT * FROM sales WHERE description LIKE '%HEART%' LIMIT 5;
```

### Your turn

Show 10 rows with a **negative quantity**. What do their invoice numbers have in common? Then count how many rows have an `invoice_no` that **starts with 'C'**.

In [ ]:
run("""

""")

<details><summary><b>Hint</b></summary>

`WHERE quantity < 0 LIMIT 10`, then `SELECT COUNT(*) ... WHERE invoice_no LIKE 'C%'`.

</details>

<details><summary><b>Answer</b></summary>

```sql
SELECT COUNT(*) AS cancellation_rows FROM sales WHERE invoice_no LIKE 'C%'
```

Result:

| cancellation_rows |
|---|
| 9288 |

**Why:** Most negative lines have invoice numbers starting with **C**: they are **cancellations** (returned or cancelled items). They are stored as separate rows with a negative quantity, so they reduce revenue when you sum. The CEO's dashboard mixes sales and cancellations; we will separate them later in the week.

> **Clue found:** About 9,300 rows are cancellations (invoice_no starts with C, negative quantity).

</details>

---
## Step 5: The biggest single line

`ORDER BY` can sort by a calculation, not only by a column: `ORDER BY quantity * unit_price DESC` puts the most
valuable lines first. You can also give the calculation a name with `AS` and show it.

### Your turn

Find the **5 most valuable product lines** (`quantity * unit_price`) in the whole data set. Show `invoice_no`, `invoice_date`, `description`, `quantity` and the line value. Does anything look suspicious?

In [ ]:
run("""

""")

<details><summary><b>Hint</b></summary>

`SELECT ..., quantity * unit_price AS line_value FROM sales ORDER BY line_value DESC LIMIT 5`

</details>

<details><summary><b>Answer</b></summary>

```sql
SELECT invoice_no, invoice_date, description, quantity, unit_price,
       ROUND(quantity * unit_price, 2) AS line_value
FROM sales
ORDER BY line_value DESC
LIMIT 5
```

Result:

| invoice_no | invoice_date | description | quantity | unit_price | line_value |
|---|---|---|---|---|---|
| 581483 | 2011-12-09 09:15:00 | PAPER CRAFT , LITTLE BIRDIE | 80995 | 2.08 | 168469.6 |
| 541431 | 2011-01-18 10:01:00 | MEDIUM CERAMIC TOP STORAGE JAR | 74215 | 1.04 | 77183.6 |
| 556444 | 2011-06-10 15:28:00 | PICNIC BASKET WICKER 60 PIECES | 60 | 649.5 | 38970.0 |
| 537632 | 2010-12-07 15:08:00 | AMAZON FEE | 1 | 13541.33 | 13541.33 |
| A563185 | 2011-08-12 14:50:00 | Adjust bad debt | 1 | 11062.06 | 11062.06 |

**Why:** The most valuable line in the whole year is **80,995 "PAPER CRAFT , LITTLE BIRDIE" on 2011-12-09**, worth about 168,000 pounds: one line that is a big part of the 9 days of December. One order of 80,995 paper birds is unusual. Remember it: tomorrow we check whether it was really sold, or cancelled.

> **Clue found:** The single biggest line of the year (80,995 paper birds, about 168,000 pounds) is on the last day, 2011-12-09.

</details>

---
## Step 6: Where do customers come from?

`DISTINCT` removes duplicate values: `SELECT DISTINCT country FROM sales` lists every country once.
`COUNT(DISTINCT column)` counts the different values.

### Your turn

How many **different countries** buy from the shop? List them in alphabetical order.

In [ ]:
run("""

""")

<details><summary><b>Hint</b></summary>

`SELECT DISTINCT country ... ORDER BY country`, and `SELECT COUNT(DISTINCT country) FROM sales`.

</details>

<details><summary><b>Answer</b></summary>

```sql
SELECT COUNT(DISTINCT country) AS n_countries FROM sales
```

Result:

| n_countries |
|---|
| 38 |

**Why:** The shop sells to 38 countries (one value is "Unspecified", a data quality issue). Most rows are from the United Kingdom. When a total moves, a change in **one big segment** (one country, one customer) is often the cause: we will break revenue down by country later.

</details>

---
## Step 7: Report to the CEO (in words)

A good update is short: **what you found, what it means, what you do next.** No SQL in the message.

### Your turn

Write **two or three sentences** to the CEO about what you know after today.

*Your answer (double-click to write):*

<details><summary><b>Hint</b></summary>

Use your clues: the period of the data, the fair comparison, and the strange rows.

</details>

<details><summary><b>Answer</b></summary>

"The 70% drop is mostly because the data for December only covers the first 9 days; compared with the same 9 days of November, the shop was even busier. The December figure also includes one unusually large order and some cancellations, which I am checking next. I will send a like-for-like revenue comparison tomorrow."

**Why:** It answers the panic first (not collapsing), says what is still unknown, and promises the next step. This is exactly what an examiner checks in a product or analytics case: a clear, calm conclusion, not just numbers.

</details>

---
## Case board after day 1

1. One row = one product line of an invoice. An invoice (order) has many rows.
2. The data ends on 2011-12-09. December has only 9 days of data, November has 30.
3. Comparing equal periods (1 to 9 November vs 1 to 9 December), December is even busier.
4. About 9,300 rows are cancellations (invoice_no starts with C, negative quantity).
5. The single biggest line of the year (80,995 paper birds, about 168,000 pounds) is on the last day, 2011-12-09.

**Tomorrow (day 2):** you learn `GROUP BY` and `SUM` to compute revenue per day and per country, compare
equal periods properly, and find out what happened to the 80,995 paper birds.

**How it's tested:** examiners give you a table and a vague business question. They watch whether you first check
what one row means and which period the data covers, before computing anything. That habit alone avoids the most
common wrong answers.

**Optional practice (about 10 minutes, auto-graded):** LeetCode SQL problems that use only today's tools:
[595 Big Countries](https://leetcode.com/problems/big-countries/),
[1757 Recyclable and Low Fat Products](https://leetcode.com/problems/recyclable-and-low-fat-products/),
[584 Find Customer Referee](https://leetcode.com/problems/find-customer-referee/) (watch out for NULL).

**Just for fun:** [SQL Murder Mystery](https://mystery.knightlab.com/) is a free detective game in the browser that
uses the same tools.